# 第 2 章：NumPy 陣列、條件篩選與訂單彙總

本 Notebook 融合 `lesson02.ipynb` 與 `practice_ch02.py`，並修復原 Notebook 中的中文編碼亂碼。

**適合對象**：具備 Python 變數與基本運算概念，準備學習 NumPy 與 Pandas 的初學者。

**學習目標**：
- 建立 NumPy 一維陣列並執行向量化運算。
- 使用比較運算建立布林遮罩並篩選資料。
- 將 Pandas 欄位轉成 NumPy 陣列計算營收。
- 篩選 2025 年已完成訂單，並按月份統計訂單數。

## 學習流程

1. 載入套件與課程資料
2. 建立 NumPy 陣列
3. 向量化計算與整除運算
4. 布林遮罩與條件篩選
5. 使用真實訂單品項計算營收
6. 篩選 2025 年已完成訂單
7. 按月份彙總訂單數
8. 練習題與常見錯誤

## 1. 環境設定與資料載入

`common.py` 提供課程共用的套件檢查與資料載入函式，因此不必在 Notebook 內重複尋找 CSV 路徑。

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

from common import ensure_packages, load_data

ensure_packages()
data = load_data()

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)

print("已載入的資料表：", list(data.keys()))

已載入的資料表： ['customers', 'products', 'orders', 'order_items', 'sessions', 'events', 'ab_assignments']


## 2. 建立 NumPy 陣列

NumPy 陣列適合儲存相同型別的數值。下面以五項商品的單價與數量為例。兩個陣列的位置必須互相對應：第一個價格配第一個數量，以此類推。

In [2]:
prices = np.array([120, 250, 80, 360, 199])
quantities = np.array([1, 2, 4, 1, 3])

print("單價：", prices)
print("數量：", quantities)
print("陣列形狀：", prices.shape)
print("元素型別：", prices.dtype)

單價： [120 250  80 360 199]
數量： [1 2 4 1 3]
陣列形狀： (5,)
元素型別： int64


## 3. 向量化運算

兩個長度相同的 NumPy 陣列相乘時，會依位置逐一計算，不必撰寫 `for` 迴圈。

In [3]:
# 各品項小計 = 單價 × 數量。
line_totals = prices * quantities
print("各品項小計：", line_totals)
print("全部品項合計：", line_totals.sum())

各品項小計： [120 500 320 360 597]
全部品項合計： 1897


### 3.1 套用折扣

陣列乘上一個數字時，NumPy 會將該數字套用到每個元素。`0.8` 代表原價的八折。

In [4]:
discounted_prices = prices * 0.8
print("八折後單價：", discounted_prices)

八折後單價： [ 96.  200.   64.  288.  159.2]


### 3.2 一般除法與整除

`/` 保留小數結果；`//` 是向下取整的整除運算。原始 Notebook 的 `prices // 5.0` 表示每個價格除以 5 後，只保留向下取整的商。

In [5]:
normal_division = prices / 5
floor_division = prices // 5

print("一般除法：", normal_division)
print("整除結果：", floor_division)

一般除法： [24.  50.  16.  72.  39.8]
整除結果： [24 50 16 72 39]


## 4. 布林遮罩與條件篩選

陣列和數值比較後，會得到一組 `True`／`False`。這組結果稱為布林遮罩，可用來選出符合條件的元素。

In [6]:
# 判斷哪些品項小計大於或等於 300。
is_large = line_totals >= 300
large_totals = line_totals[is_large]

print("布林遮罩：", is_large)
print("符合條件的小計：", large_totals)

# 也可以將比較條件直接寫在中括號內。
print("直接篩選：", line_totals[line_totals >= 300])

布林遮罩： [False  True  True  True  True]
符合條件的小計： [500 320 360 597]
直接篩選： [500 320 360 597]


**重點**：布林遮罩的長度必須與被篩選的陣列相同，而且每一個 `True` 都會保留相同位置的元素。

## 5. 使用真實訂單品項計算營收

`order_items` 是 Pandas DataFrame。使用 `.to_numpy()` 可將欄位轉成 NumPy 陣列，再套用相同的向量化計算。

每筆品項營收公式：

`數量 × 單價 × (1 - 折扣率)`

In [7]:
order_items = data["order_items"].copy()

quantity = order_items["quantity"].to_numpy()
unit_price = order_items["unit_price"].to_numpy()
discount_rate = order_items["discount_rate"].to_numpy()

# NumPy 會逐列計算每筆訂單品項的折扣後營收。
line_revenue = quantity * unit_price * (1 - discount_rate)

print(f"品項數量：{len(line_revenue):,}")
print(f"平均品項營收：{line_revenue.mean():,.2f}")
print(f"最高品項營收：{line_revenue.max():,.2f}")
print(f"最低品項營收：{line_revenue.min():,.2f}")

品項數量：39,627
平均品項營收：2,767.15
最高品項營收：14,925.00
最低品項營收：100.80


### 5.1 篩選高營收品項

延續布林遮罩概念，統計營收大於或等於指定門檻的品項數量與比例。

In [8]:
threshold = 500
high_value_mask = line_revenue >= threshold
high_value_count = high_value_mask.sum()
high_value_ratio = high_value_mask.mean()

print(f"品項營收 >= {threshold:,} 的數量：{high_value_count:,}")
print(f"占全部品項比例：{high_value_ratio:.2%}")

品項營收 >= 500 的數量：37,848
占全部品項比例：95.51%


為方便檢查，也可以把計算結果加回 DataFrame，再顯示少量資料。

In [9]:
order_items["line_revenue"] = line_revenue
display(
    order_items.loc[
        order_items["line_revenue"] >= threshold,
        ["order_id", "quantity", "unit_price", "discount_rate", "line_revenue"],
    ].head()
)

,order_id,quantity,unit_price,discount_rate,line_revenue
0,1,1,567,0.05,538.65
1,2,1,3850,0.05,"3,657.50"
2,2,1,778,0.10,700.20
3,3,1,2193,0.00,"2,193.00"
4,3,1,1580,0.15,"1,343.00"


## 6. 篩選 2025 年已完成訂單

接著融合 `practice_ch02.py` 的練習。先複製訂單資料，再將 `order_date` 轉成日期型別，才能使用 `.dt.year` 取得年份。

篩選條件有兩個：
- 訂單年份等於 2025。
- 訂單狀態等於 `completed`。

兩個條件必須同時成立，所以使用 `&`。

In [10]:
orders = data["orders"].copy()
orders["order_date"] = pd.to_datetime(orders["order_date"])

is_2025 = orders["order_date"].dt.year == 2025
is_completed = orders["status"] == "completed"

completed_2025 = orders.loc[is_2025 & is_completed].copy()

print(f"2025 年已完成訂單：{len(completed_2025):,} 筆")
display(completed_2025.head())

2025 年已完成訂單：10,106 筆


,order_id,customer_id,order_date,status,payment_type
0,1,2323,2025-05-02,completed,wallet
5,6,2005,2025-01-14,completed,card
10,11,2091,2025-09-01,completed,atm
11,12,1710,2025-02-15,completed,card
13,14,1135,2025-05-19,completed,card


### 為什麼每個條件都要加括號？

Pandas 使用 `&` 表示逐列的「而且」。如果直接把完整條件寫在一起，每一個比較式都應加括號：

```python
(orders["order_date"].dt.year == 2025) & (orders["status"] == "completed")
```

不能使用一般 Python 的 `and`，因為 `and` 無法逐列處理整個 Series。

## 7. 按月份統計已完成訂單

使用 `.dt.to_period("M")` 將每日日期轉成月份，例如 `2025-03-18` 會轉成 `2025-03`。接著依月份分組並計算每組筆數。

In [11]:
completed_2025["order_month"] = completed_2025["order_date"].dt.to_period("M")

monthly_counts = (
    completed_2025.groupby("order_month")
    .size()
    .rename("completed_orders")
)

print("2025 年各月已完成訂單數：")
display(monthly_counts.to_frame())

2025 年各月已完成訂單數：


,completed_orders
order_month,
2025-01,845
2025-02,750
2025-03,859
2025-04,866
2025-05,857
2025-06,814
2025-07,848
2025-08,901
2025-09,832


### 程式鏈的意思

- `groupby("order_month")`：將相同月份的資料放在同一組。
- `.size()`：計算每一組有幾列，也就是訂單筆數。
- `.rename("completed_orders")`：替統計結果設定容易理解的名稱。

## 8. 完整性檢查

每月訂單數加總後，應等於 `completed_2025` 的總筆數。這是一個簡單但實用的資料驗證方式。

In [12]:
monthly_total = int(monthly_counts.sum())
filtered_total = len(completed_2025)

print(f"每月統計加總：{monthly_total:,}")
print(f"篩選結果總數：{filtered_total:,}")
print("兩者是否一致：", monthly_total == filtered_total)

每月統計加總：10,106
篩選結果總數：10,106
兩者是否一致： True


## 9. 練習題

請統計 2025 年每個月的「所有訂單數」，不限制訂單狀態，並與已完成訂單數比較。

思考：哪一個月份的未完成訂單最多？

In [13]:
# TODO：可先遮住以下參考答案，再自行完成。
orders_2025 = orders.loc[orders["order_date"].dt.year == 2025].copy()
orders_2025["order_month"] = orders_2025["order_date"].dt.to_period("M")

all_monthly_counts = orders_2025.groupby("order_month").size().rename("all_orders")
comparison = pd.concat([all_monthly_counts, monthly_counts], axis=1).fillna(0).astype(int)
comparison["not_completed"] = comparison["all_orders"] - comparison["completed_orders"]
display(comparison)

,all_orders,completed_orders,not_completed
order_month,,,
2025-01,897,845,52
2025-02,796,750,46
2025-03,915,859,56
2025-04,928,866,62
2025-05,925,857,68
2025-06,881,814,67
2025-07,915,848,67
2025-08,968,901,67
2025-09,890,832,58


## 常見錯誤與延伸

**常見錯誤**：
- 兩個 NumPy 陣列長度不同，無法依位置運算。
- 日期仍是字串時就使用 `.dt.year`，會發生錯誤；應先用 `pd.to_datetime()`。
- Pandas 多條件篩選使用 `and`；正確方式是各條件加括號後使用 `&`。
- 在篩選結果上直接賦值而未使用 `.copy()`，可能出現 `SettingWithCopyWarning`。

**延伸練習**：
- 計算每月已完成訂單占所有訂單的比例。
- 將 `monthly_counts` 畫成長條圖。
- 分別比較 `completed`、`cancelled` 等狀態的每月趨勢。

## 重點整理

- NumPy 向量化運算能一次處理整組資料。
- 比較運算會產生布林遮罩，可用於 NumPy 與 Pandas 篩選。
- Pandas 日期欄位必須先轉成日期型別，才能使用 `.dt`。
- `groupby()` 搭配 `.size()` 可計算每個群組的資料筆數。